### This notebook format data for training deepSCENIC from a SCENIC+ object

In [1]:
import pickle
import scanpy as sc
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

Read the SCENIC+ object

In [8]:
import dill
scplus_folder = "/lustre1/project/stg_00002/lcb/gpartel/projects/deepSCENIC/data/DPCL/"
scenicpls_obj = dill.load(
        open(os.path.join(scplus_folder, 'scplus_obj.pkl'), 'rb'))

In [9]:
# PATH WHERE TO SAVE THE DATA
DATA_PATH = '/staging/leuven/stg_00002/lcb/gpartel/projects/deepSCENIC/data/DPCL/'

# Save cell metadata
scenicpls_obj.metadata_cell.to_csv(DATA_PATH+'cells_metadata.csv')

Extract RNA and ATAC data, preprocessing and filtering

In [4]:
ad_rna = sc.AnnData(scenicpls_obj.to_df('EXP'))
ad_rna

AnnData object with n_obs × n_vars = 4000 × 25651

In [5]:
sc.pp.filter_genes(ad_rna, min_cells=10)
ad_rna

AnnData object with n_obs × n_vars = 4000 × 25651
    var: 'n_cells'

In [6]:
sc.pp.normalize_total(ad_rna)
sc.pp.log1p(ad_rna)
ad_rna.layers['log_norm'] = ad_rna.X

In [7]:
# # Filter regions based on minimum number of fragments
# min_nr_acc = 20
# scenicpls_obj.subset(regions = scenicpls_obj.metadata_regions[scenicpls_obj.metadata_regions.cisTopic_nr_acc>min_nr_acc].index.to_list())

In [10]:
ad_atac = sc.AnnData(scenicpls_obj.to_df('ACC').T, obs=scenicpls_obj.metadata_cell)
ad_atac

/tmp/ipykernel_2137442/1309858748.py:1: FutureWarning: X.dtype being converted to np.float32 from int32. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  ad_atac = sc.AnnData(scenicpls_obj.to_df('ACC').T, obs=scenicpls_obj.metadata_cell)


AnnData object with n_obs × n_vars = 4000 × 589734
    obs: 'GEX_sample_id', 'GEX_leiden', 'ACC_cisTopic_nr_frag', 'ACC_cisTopic_log_nr_frag', 'ACC_cisTopic_nr_acc', 'ACC_cisTopic_log_nr_acc', 'ACC_sample_id', 'ACC_Log_total_nr_frag', 'ACC_Log_unique_nr_frag', 'ACC_Total_nr_frag', 'ACC_Unique_nr_frag', 'ACC_Dupl_nr_frag', 'ACC_Dupl_rate', 'ACC_Total_nr_frag_in_regions', 'ACC_Unique_nr_frag_in_regions', 'ACC_FRIP', 'ACC_TSS_enrichment', 'ACC_barcode', 'ACC_Cell_type', 'ACC_pycisTopic_leiden_10_0.3', 'ACC_Line_type'

In [11]:
regs = []
for ct in tqdm(ad_atac.obs.ACC_Cell_type.unique()):
    ad_ct = ad_atac[ad_atac.obs.ACC_Cell_type==ct].copy()
    sc.pp.filter_genes(ad_ct, min_cells=ad_ct.shape[0]*0.10)
    regs.append(ad_ct.var_names)
regs = [x.to_list() for x in regs]    
regs = list(set(sum(regs,[])))
ad_atac = ad_atac[:, regs]
ad_atac

100%|██████████| 8/8 [00:31<00:00,  3.97s/it]


View of AnnData object with n_obs × n_vars = 4000 × 563649
    obs: 'GEX_sample_id', 'GEX_leiden', 'ACC_cisTopic_nr_frag', 'ACC_cisTopic_log_nr_frag', 'ACC_cisTopic_nr_acc', 'ACC_cisTopic_log_nr_acc', 'ACC_sample_id', 'ACC_Log_total_nr_frag', 'ACC_Log_unique_nr_frag', 'ACC_Total_nr_frag', 'ACC_Unique_nr_frag', 'ACC_Dupl_nr_frag', 'ACC_Dupl_rate', 'ACC_Total_nr_frag_in_regions', 'ACC_Unique_nr_frag_in_regions', 'ACC_FRIP', 'ACC_TSS_enrichment', 'ACC_barcode', 'ACC_Cell_type', 'ACC_pycisTopic_leiden_10_0.3', 'ACC_Line_type'

In [12]:
ad_atac.region_names = ad_atac.var_names

In [13]:
TFs = pd.read_csv("/staging/leuven/stg_00002/lcb/gpartel/software/DeepSEM_multimodal/resources/allTFs_hg38.txt", header=None)[0].to_list()
# Subset TFs present in RNA data
TFs = ad_rna.var_names[ad_rna.var_names.isin(TFs)]
len(TFs)

1594

Get regions per gene (up to 150 kb up and down stream of the gene)

In [14]:
from pycisTopic.utils import region_names_to_coordinates, coord_to_region_names
from scenicplus.enhancer_to_gene import get_search_space
import pyranges as pr

max_dist = 1000000
search_space = get_search_space(ad_atac, pr_annot = pr.PyRanges(region_names_to_coordinates(ad_atac.region_names)),
                                species = 'hsapiens',
                                assembly = 'hg38',
                                upstream = [max_dist, max_dist],
                                downstream = [max_dist, max_dist],
                                inplace=False,
                               )

2023-12-13 09:32:22,190 R2G          INFO     Downloading gene annotation from biomart dataset: hsapiens_gene_ensembl
2023-12-13 09:32:38,326 R2G          INFO     Downloading chromosome sizes from: http://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.chrom.sizes
2023-12-13 09:32:40,047 R2G          INFO     Extending promoter annotation to 10 bp upstream and 10 downstream
2023-12-13 09:32:42,639 R2G          INFO     Extending search space to:
            						1000000 bp downstream of the end of the gene.
            						1000000 bp upstream of the start of the gene.
2023-12-13 09:32:55,850 R2G          INFO     Intersecting with regions.


join: Strand data from other will be added as strand data to self.
If this is undesired use the flag apply_strand_suffix=False.
To turn off the warning set apply_strand_suffix to True or False.


2023-12-13 09:33:00,889 R2G          INFO     Calculating distances from region to gene
2023-12-13 10:30:49,022 R2G          INFO     Done!


Building region to gene distance matrix

In [41]:
search_space['Abs_distance'] = np.array([abs(x[0]) + 1 for x in search_space['Distance'].to_list()]).astype(int)

In [42]:
search_space_subset = search_space.loc[[gene in ad_rna.var_names for gene in search_space['Gene']]]

In [43]:
search_space_subset_test = search_space_subset[(search_space_subset.Name.str.startswith('chr18')) | (search_space_subset.Name.str.startswith('chr11')) | (search_space_subset.Name.str.startswith('chr7'))]
regions_test = search_space_subset_test.loc[:, 'Name'].unique().tolist()
genes_test = search_space_subset_test.loc[:, 'Gene'].unique().tolist()

In [44]:
## Split train/validation
search_space_subset_train = search_space_subset[(~search_space_subset.Name.str.startswith('chr18')) & (~search_space_subset.Name.str.startswith('chr11')) & (~search_space_subset.Name.str.startswith('chr7'))]
regions_train = search_space_subset_train.loc[:, 'Name'].unique().tolist()
genes_train = search_space_subset_train.loc[:, 'Gene'].unique().tolist()

In [45]:
len(regions_train)

484247

In [46]:
len(regions_test)

74068

In [47]:
search_space_subset_test = search_space_subset_test[search_space_subset_test.Name.isin(regions_test)]
search_space_subset_train = search_space_subset_train[search_space_subset_train.Name.isin(regions_train)]

In [48]:
r2gdist_train = search_space_subset_train.pivot(index='Name', columns='Gene', values='Abs_distance')
r2gdist_test = search_space_subset_test.pivot(index='Name', columns='Gene', values='Abs_distance')

/tmp/ipykernel_2137442/26681771.py:1: PerformanceWarning: The following operation may generate 6517964620 cells in the resulting pandas object.
  r2gdist_train = search_space_subset_train.pivot(index='Name', columns='Gene', values='Abs_distance')


In [49]:
# Scale matrix between 0 and 1
r2gdist_train = r2gdist_train / max_dist
r2gdist_train = r2gdist_train.fillna(0)

r2gdist_test = r2gdist_test / max_dist
r2gdist_test = r2gdist_test.fillna(0)

In [50]:
# Add missing TFs to the datasets
r2gdist_train.loc[:, TFs[~np.isin(TFs, r2gdist_train.columns)]] = 0
r2gdist_test.loc[:, TFs[~np.isin(TFs, r2gdist_test.columns)]] = 0

/tmp/ipykernel_2137442/1187292137.py:2: FutureWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  r2gdist_train.loc[:, TFs[~np.isin(TFs, r2gdist_train.columns)]] = 0
/tmp/ipykernel_2137442/1187292137.py:3: FutureWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  r2gdist_test.loc[:, TFs[~np.isin(TFs, r2gdist_test.columns)]] = 0


In [51]:
# Put TFs first
r2gdist_train = r2gdist_train.loc[:, TFs.to_list() + r2gdist_train.columns[~r2gdist_train.columns.isin(TFs)].to_list()]
r2gdist_test = r2gdist_test.loc[:, TFs.to_list() + r2gdist_test.columns[~r2gdist_test.columns.isin(TFs)].to_list()]

In [52]:
# Ensure same order of indices
regions_train = r2gdist_train.index.values.tolist()
genes_train = r2gdist_train.columns.values.tolist()

regions_test = r2gdist_test.index.values.tolist()
genes_test = r2gdist_test.columns.values.tolist()

ad_atac_train = ad_atac[:, regions_train].copy()
ad_rna_train = ad_rna[:, genes_train].copy()
ad_atac_test = ad_atac[:, regions_test].copy()
ad_rna_test = ad_rna[:, genes_test].copy()

Make region to gene matrix sparse

In [53]:
from scipy.sparse import coo_array, save_npz
r2gdist_coo_train = coo_array(r2gdist_train)
r2gdist_coo_test = coo_array(r2gdist_test)

In [33]:
ad_rna_train.write(DATA_PATH+'raw_exprMat_train.h5ad')
ad_atac_train.write(DATA_PATH+'fragment_matrix_train.h5ad')
ad_rna_test.write(DATA_PATH+'raw_exprMat_test.h5ad')
ad_atac_test.write(DATA_PATH+'fragment_matrix_test.h5ad')
save_npz(DATA_PATH+'r2gdist_train.npz', r2gdist_coo_train)
save_npz(DATA_PATH+'r2gdist_test.npz', r2gdist_coo_test)
with open(DATA_PATH+'TFs.txt', 'w') as f:
    for item in TFs:
        f.write("%s\n" % item)   

... storing 'GEX_sample_id' as categorical
... storing 'ACC_sample_id' as categorical
... storing 'ACC_Cell_type' as categorical
... storing 'ACC_Line_type' as categorical
... storing 'GEX_sample_id' as categorical
... storing 'ACC_sample_id' as categorical
... storing 'ACC_Cell_type' as categorical
... storing 'ACC_Line_type' as categorical


AnnData object with n_obs × n_vars = 4000 × 484247
    obs: 'GEX_sample_id', 'GEX_leiden', 'ACC_cisTopic_nr_frag', 'ACC_cisTopic_log_nr_frag', 'ACC_cisTopic_nr_acc', 'ACC_cisTopic_log_nr_acc', 'ACC_sample_id', 'ACC_Log_total_nr_frag', 'ACC_Log_unique_nr_frag', 'ACC_Total_nr_frag', 'ACC_Unique_nr_frag', 'ACC_Dupl_nr_frag', 'ACC_Dupl_rate', 'ACC_Total_nr_frag_in_regions', 'ACC_Unique_nr_frag_in_regions', 'ACC_FRIP', 'ACC_TSS_enrichment', 'ACC_barcode', 'ACC_Cell_type', 'ACC_pycisTopic_leiden_10_0.3', 'ACC_Line_type'

In [1]:
import scanpy as sc

DATA_PATH = '/lustre1/project/stg_00002/lcb/gpartel/projects/deepSCENIC/data/DPCL/'
ad_rna_train = sc.read(DATA_PATH+'raw_exprMat_train.h5ad')
ad_rna_test = sc.read(DATA_PATH+'raw_exprMat_test.h5ad')

ad_atac_train = sc.read(DATA_PATH+'fragment_matrix_train.h5ad')
ad_atac_test = sc.read(DATA_PATH+'fragment_matrix_test.h5ad')

In [2]:
# Merge train and test dataset putting non TFs test genes at the end
ad_rna = sc.concat((ad_rna_train, ad_rna_test[:, ~ad_rna_test.var_names.isin(ad_rna_train.var_names)]), axis=1)
ad_rna

AnnData object with n_obs × n_vars = 4000 × 15310
    var: 'n_cells'
    layers: 'log_norm'

In [3]:
ad_atac = sc.concat((ad_atac_train, ad_atac_test), axis=1)
ad_atac

AnnData object with n_obs × n_vars = 4000 × 558315

In [4]:
ad_rna.write(DATA_PATH+'raw_exprMat.h5ad')
ad_atac.write(DATA_PATH+'fragment_matrix.h5ad')